# I-01: recovery after a changed winning value

Two agents collaborate on a maximum. After one value changes, compare retaining
their conversation with clearing it. This is a controlled recovery extension of
SILO-BENCH I-01: six qualification worlds, three pilots, twenty main worlds.

**Run one stage cell at a time. Pressing Run performs the named stage.**
No widget forms or extra action buttons. **Do not use Run All / Save & Run All:**
those commands can execute inference and multiple stages. Each research cell
requests only its displayed bounded scope; successful work starts no later stage.

Attach the matching **private source dataset**, select **GPU T4 ×2** and enable
**Internet**. Actual host/runtime compatibility is checked before installation.
Model qualification and research outcomes remain unverified until executed.

Setup → qualification → pilot 1/review → pilot 2/review → pilot 3/review →
freeze → prepare/run/review main batches → report → portable archive.

The whole study has a **500-new-physical-attempt ceiling**. Diagnostics, failures
and eligible retries count. Historical 45 are separate. First-attempt ceilings
exclude eligible retries, which consume shared 28 contingency and study allowance.

Models stay resident during inspection and between stages. Interrupt requests
pause; an admitted call must settle within its bound. Recovery recipes are below.


## 1. Configure and inspect the workspace

Edit actual session facts here. A unique verified source ZIP/expanded directory is selected automatically; set SOURCE_BUNDLE only when ambiguous. Inventory only; no installation or generation.


In [1]:
# Supply CURRENT session facts before loading models; no duration/quota is guessed.
START_MODE = "new"  # "reopen" for intact files; "restore" for a selected archive.
SESSION_DEADLINE_UTC = None  # Actual future ISO time: YYYY-MM-DDTHH:MM:SS+00:00.
SESSION_ALLOWANCE_EVIDENCE = ""  # Describe the actual allowance shown by Kaggle.

# Only for restoration: uncomment and supply exact checkpoint/tail evidence.
# RESTORE_CHECKPOINT = "/kaggle/input/EXACT-DOWNLOADED-CHECKPOINT"
# RESTORE_TAIL = "latest"  # "unknown" if the latest safe boundary is uncertain.
# RESTORE_EVIDENCE = "Actual evidence about the saved boundary and prior session."

from pathlib import Path
import hashlib, json, shutil, sys, tempfile, zipfile

# Select one exact small source attachment. ZIP and Kaggle-expanded roots work.
SOURCE_BUNDLE = None  # Set only if more than one candidate is attached.
source_candidates = list(Path('/kaggle/input').rglob('i01-source.zip'))
source_candidates += [p.parent for p in Path('/kaggle/input').rglob('source-release.json')]
if SOURCE_BUNDLE is None:
    if len(source_candidates) != 1:
        raise RuntimeError(f'Select SOURCE_BUNDLE explicitly; candidates: {source_candidates}')
    SOURCE_BUNDLE = source_candidates[0]
selected_source = Path(SOURCE_BUNDLE)
with tempfile.TemporaryDirectory(prefix='i01-source-', dir='/kaggle/working') as staging_name:
    staging = Path(staging_name)
    if selected_source.is_file():
        with zipfile.ZipFile(selected_source) as source_zip:
            infos = source_zip.infolist()
            if sum(p.file_size for p in infos) > 64 * 2**20 or len({p.filename for p in infos}) != len(infos):
                raise RuntimeError('Source attachment size or duplicate member violation')
            for item in infos:
                p = Path(item.filename)
                if p.is_absolute() or '..' in p.parts or '\\' in item.filename or ':' in item.filename or ((item.external_attr >> 16) & 0o170000) == 0o120000:
                    raise RuntimeError('Unsafe source archive member')
            if source_zip.testzip() is not None:
                raise RuntimeError('Source ZIP integrity failed')
            source_zip.extractall(staging)
    elif selected_source.is_dir():
        if any(p.is_symlink() for p in selected_source.rglob('*')):
            raise RuntimeError('Select an ordinary verified expanded source directory')
        shutil.copytree(selected_source, staging, dirs_exist_ok=True)
    else:
        raise RuntimeError('Missing exact selected source attachment')
    release = json.loads((staging/'source-release.json').read_text())
    if release['format'] != 'i01-online-source-v1':
        raise RuntimeError('Wrong source release format')
    head = release.get('git_head', '')
    if len(head) != 40 or any(c not in '0123456789abcdef' for c in head):
        raise RuntimeError('Invalid source release commit')
    source_root = Path('/kaggle/working') / ('i01-source-' + head[:12])
    for relative, wanted in release['files'].items():
        path = staging/relative
        if not path.is_file() or path.stat().st_size != wanted['bytes'] or hashlib.sha256(path.read_bytes()).hexdigest() != wanted['sha256']:
            raise RuntimeError(f'Source identity failed: {relative}')
        if source_root.exists() and (not (source_root/relative).is_file() or hashlib.sha256((source_root/relative).read_bytes()).hexdigest() != wanted['sha256']):
            raise RuntimeError('Existing source differs; select a fresh session/source directory')
    if not source_root.exists():
        shutil.copytree(staging, source_root)
loaded_source = sys.modules.get('recovery')
if loaded_source is not None and not Path(loaded_source.__file__).resolve().is_relative_to(source_root.resolve()):
    raise RuntimeError('Restart the notebook kernel before using this source release; saved study files are preserved.')
sys.path.insert(0, str(source_root))
from recovery.i01.notebook import Study
if 'study' not in globals():
    study = Study(source_root)
study.bootstrap_status()


Workspace,Observed value
Python,"3.13.15 (main, Aug 6 2026, 11:06:22) [GCC 13.3.0]"
System,Linux
Source,/kaggle/working/i01-source-659c056abfd1
Study files,/kaggle/working/i01-study


Device,Model,UUID,Memory (MiB)
0,Tesla T4,GPU-c6226d0b-b473-48d6-3019-178aeaed74f8,15360
1,Tesla T4,GPU-02e13583-1987-2320-c6e4-1947a83faf4f,15360


Storage,Path,Free GiB
scratch,/kaggle/temp/i01,1069.91
working,/kaggle/working,19.5


Next: run Install the pinned runtime. Host acceptance still occurs before installation.


## 2. Install the pinned runtime

Create/recover the isolated environment without ensurepip, bootstrap verified pinned pip and install the unchanged runtime lock; fetch metadata/tokenizer. Live progress and /kaggle/working/i01-setup/ logs. No model loading or generation.


In [2]:
study.install_runtime();


Setup: venv
Setup: download verified pip bootstrap wheel
Setup: bootstrap-pip
Setup: pip-isolation
Setup: dependencies
dependencies: 15s elapsed;      ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 338.7 MB/s  0:00:01
dependencies: 30s elapsed;      ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 375.9 MB/s  0:00:00
dependencies: 45s elapsed;   Downloading sniffio-1.3.1-py3-none-any.whl (10 kB)
dependencies: 60s elapsed;   Downloading torch-2.11.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl (820.3 MB)
dependencies: 75s elapsed; py, anyio, rich, requests-toolbelt, pydantic, nvidia-cusolver-cu12, matplotlib, httpx2, httpx, typer, langsmith, torch, langchain-core, huggingface-hub, tokenizers, langgraph-sdk, langgraph-checkpoint, bitsandbytes, accelerate, transformers, langgraph-prebuilt, langgraph-checkpoint-sqlite, langgraph
dependencies: 90s elapsed; py, anyio, rich, requests-toolbelt, pydantic, nvidia-cusolver-cu12, matplotlib, httpx2, httpx, typer, langsmith, torch,

Measurement,Recorded value
Interpreter,/kaggle/temp/i01/i01-venv/bin/python
Source,/kaggle/working/i01-source-659c056abfd1
Scratch,/kaggle/temp/i01
Working,/kaggle/working
Hf cache,/kaggle/temp/i01/hf-cache
Setup seconds,179.00569716100003
Model attempts,0


## 3. Open the study

Create fixed inputs, reopen intact files or explicitly restore into a fresh writable directory. Opening and inspection need no model allocation or session window. Conflicting studies are never overwritten.


In [3]:
study.open(START_MODE,
    selected=globals().get("RESTORE_CHECKPOINT"),
    tail=globals().get("RESTORE_TAIL"), evidence=globals().get("RESTORE_EVIDENCE", ""));


session: recorded in the stage receipt


Measurement,Recorded value
Attempts,0
Attempts unknown generated tokens,0
Attempts unknown ready seconds,0
Clear minus retain,Unknown / not recorded
Complete case descriptive mean,Unknown / not recorded
Contingency,0
Correct prefix denominator,0
Correct prefix interval,Unknown / not recorded
Correct prefix mean,Unknown / not recorded
Cost complete pairs,0


Main coverage,Value
Known paired outcomes,0 / 20
New physical attempts,0 / 500
Clear minus retain,Unknown / not recorded
Unknown-outcome bounds,"[-1.0, 1.0]"


Logical slot,Resident,Load receipt
0,No,Not recorded
1,No,Not recorded


Budget,Value
attempts,0
contingency,0
in flight,0
lost spending,"{'resolved': True, 'upper': 0}"
remaining,500


## 4. Check the implementation

Lightweight mathematics/private-view/lifecycle/durability/accounting/archive/restore and native-tokenizer checks. Temporary scripted records, zero model attempts.


In [4]:
study.check_implementation();


Checkpoint action/selfcheck: 1,238,884 bytes; integrity passed
selfcheck: complete · checkpoint /kaggle/working/i01-study/exports/latest.zip


Measurement,Recorded value
Environment,scripted temporary DBs; no model loaded
Executed at,2026-10-14T18:07:28.658012+00:00
Model attempts,0
Native tokenizer,Yes
Passed,Yes


Lightweight check,State
native fixture 827/revised 780; duplicates/negative/boundary,Passed
fixed strata/seeds; private views; finality/wait; literal siblings; invalid actions,Passed
raw-before-effects; idempotence; pre-persistence retry; 499/500; backup/ZIP/expanded restore; unknown vs zero,Passed


Native tokenizer: passed


Checkpoint: passed


## 5. Download verified weights

Retrieve/reuse the pinned Qwen3.5-9B files; verify native identities and actual available space. Progress appears below. No inference.


In [5]:
study.download_weights();


Setup: weights-command
weights-command: 150s elapsed; Fetching 12 files: 100%|██████████| 12/12 [01:23<00:00,  6.93s/it]


Measurement,Recorded value
Model,Qwen/Qwen3.5-9B
Revision,c202236235762e1c871ad0ccb60c8ee5ba337b9a
Snapshot,/kaggle/temp/i01/hf-cache/models--Qwen--Qwen3.5-9B/snapshots/c202236235762e1c871ad0ccb60c8ee5ba337b9a
Native total bytes,19329302519
Weights,Yes
Model attempts,0


## 6. Load both T4 workers

Record this session's actual future deadline and allowance evidence from the configuration cell, then load/attach two resident replicas and show device/load receipts. Missing facts stop before loading. No hidden warmup. Inspection does not unload them.


In [6]:
study.set_session_window(SESSION_DEADLINE_UTC, SESSION_ALLOWANCE_EVIDENCE);
study.load_models();


Stopped window: Set the actual UTC session deadline and allowance evidence in the configuration cell.. Saved evidence remains available through study.inspect() and study.links().


ValueError: Set the actual UTC session deadline and allowance evidence in the configuration cell.

## 7. Check fresh-state readiness

Exactly six short A/B/A diagnostics across both workers: ≤6 first attempts plus eligible contingency. A failed gate stops the cell with saved evidence.


In [ ]:
study.check_readiness();


## 8. Check direct-mode competence

Twelve original/revised controls, ≤12 first attempts plus eligible contingency. Require ≥11/12 correct and zero malformed/truncated actions. Failure stops here; inspect it below.


In [ ]:
study.qualify_direct();


## 9. Inspect qualification

Read-only control outcomes and complete saved evidence. If direct passed, proceed to headroom. A protocol defect blocks research; only a documented competence limitation enables the optional thinking screen.


In [ ]:
study.inspect_qualification();


## Optional: thinking qualification

Skip if direct passed; running this cell then makes zero calls. Otherwise supply your evidence-based competence reason only after direct failure without protocol errors. ≤12 first attempts; no automatic fallback or extra model candidate.


In [ ]:
THINKING_COMPETENCE_REASON = None
study.qualify_thinking(THINKING_COMPETENCE_REASON);


## 10. Check selected-mode headroom

The passing recorded mode is derived automatically. Two full-input/full-output probes, ≤2 first attempts. Require ≥1.5 GiB measured free headroom on each T4. Output forcing applies only to diagnostics. Restored/reloaded sessions use the separately charged reload recipe.


In [ ]:
study.check_headroom();


## 11. Run pilot 1

Run **pilot 1 only**, including original team, literal retain/clear siblings and original/revised controls. ≤20 first attempts plus eligible contingency. Conversation, outcomes, costs and cumulative checkpoint appear automatically. Inspect/download before review; recovery errors alone do not fail the gate.


In [ ]:
study.run_pilot(1);


## Review pilot 1

Replace the None findings with actual True/False conclusions and write trace-based notes. Useful original communication is a human finding, not inferred from correct answers. Records only this review; no next pilot/freeze. Unreviewed never means approved.


In [ ]:
study.record_pilot_review(1,
    useful_communication=None,
    usable=None,
    notes="");


## 12. Run pilot 2

Run **pilot 2 only**, including original team, literal retain/clear siblings and original/revised controls. ≤20 first attempts plus eligible contingency. Conversation, outcomes, costs and cumulative checkpoint appear automatically. Inspect/download before review; recovery errors alone do not fail the gate.


In [ ]:
study.run_pilot(2);


## Review pilot 2

Replace the None findings with actual True/False conclusions and write trace-based notes. Useful original communication is a human finding, not inferred from correct answers. Records only this review; no next pilot/freeze. Unreviewed never means approved.


In [ ]:
study.record_pilot_review(2,
    useful_communication=None,
    usable=None,
    notes="");


## 13. Run pilot 3

Run **pilot 3 only**, including original team, literal retain/clear siblings and original/revised controls. ≤20 first attempts plus eligible contingency. Conversation, outcomes, costs and cumulative checkpoint appear automatically. Inspect/download before review; recovery errors alone do not fail the gate.


In [ ]:
study.run_pilot(3);


## Review pilot 3

Replace the None findings with actual True/False conclusions and write trace-based notes. Useful original communication is a human finding, not inferred from correct answers. Records only this review; no next pilot/freeze. Unreviewed never means approved.


In [ ]:
study.record_pilot_review(3,
    useful_communication=None,
    usable=None,
    notes="");


## 14. Review the main configuration

Read qualification, three pilot reviews, exact configuration and measured attempt/time forecast. This cell neither freezes nor generates.


In [ ]:
study.review_configuration();


## 15. Freeze for main

Write actual review notes, then run to freeze accepted configuration. Gates require correct original teams/controls, valid protocol, documented communication/usability and adequate resources. Download the pre-main checkpoint. No main calls.


In [ ]:
FREEZE_NOTES = ""
study.freeze_configuration(FREEZE_NOTES);


## 16. Prepare the next main batch

Prepare the next fixed two worlds, or return the existing incomplete scope. Change count to 1 explicitly for a one-world batch. Show IDs, remaining work, ≤19/38 first-attempt ceiling, eligible retry allowance and forecast. No generation. Repeat sections 16–18 only by choice.


In [ ]:
study.prepare_main_batch(count=2);


## 17. Run the prepared main batch

Execute/resume only the saved scope: ≤19/38 first attempts plus eligible contingency. No next-batch preparation/execution. A completed saved batch displays results with zero calls. Interrupt requests pause; wait for settlement before export/closure.


In [ ]:
study.run_main_batch();


## 18. Inspect and acknowledge the batch

Inspect the previous output and complete conversation files; download the checkpoint, then run this acknowledgement. Saved batch ID is derived automatically; extra notes are optional. No claim of verified understanding/download. Return to section 16 or continue to reporting when all intended work is recorded.


In [ ]:
study.record_batch_review(notes="");


## 19. Build the final or partial report

Read-only canonical reconstruction: twenty intended worlds, exact paired outcomes, unknown bounds, correct-prefix descriptive stratum, costs, failures, figures and limitations. No inference/reload. Download HTML/text/JSONL, CSV/plotting data and SVG/PNG.


In [ ]:
study.build_report();


## 20. Save and verify the final archive

Quiescent snapshot and exact fresh-directory analysis comparison, zero model calls. Save ZIP outside Kaggle. Save/download the executed notebook separately **without Save & Run All**. Links do not prove backup.


In [ ]:
study.save_checkpoint(verify=True);


## Finish the session

Run after saving the checkpoint and executed notebook. Close models/controller, then stop Kaggle through its UI. Process exit does not establish provider deallocation; partial outcomes remain partial.


In [ ]:
study.finish_session();


## Recovery and support

Copy the needed ordinary commands into a cell. These recipes do not execute
merely by opening this appendix. Use the action matching saved state; never reset
a study to work around a blocked gate.

### Status, pause and a safe checkpoint

    study.show_status();
    study.pause();  # Request only; an admitted call may still be active.
    # Wait for settlement / confirmed bounded termination, then:
    study.save_checkpoint();

Interrupting a generation cell already requests pause. Do not infer process death
from a stale PID or claim interrupt immediately produced a safe checkpoint.

### Reopen or restore

For intact files after kernel restart, rerun workspace/runtime setup as needed,
set START_MODE = "reopen" and fresh actual session deadline/evidence, then Open
the study. The ownership lock refuses a second live controller.

For another session, attach the **matching source release** and exact downloaded
checkpoint. Set START_MODE = "restore"; uncomment the restore values in the
configuration cell and record genuine latest-boundary or unknown-tail evidence.
Restore into a fresh writable directory. ZIP and expanded roots both work.
No archive is chosen by timestamp; a missing ZIP never silently creates a study.

After reopening/restoring: self-check, reuse/download weights, then load workers.
New session/replacement workers need **two contingency-charged probes**:

    study.check_headroom(reload=True);
    study.run_main_batch();  # Resume only a saved interrupted main scope.

Do not redo already accepted readiness/qualification just to restore the study.
Completed calls reuse durable raw replies without generation. Load/reload evidence
remains recorded; diagnostics, retries and warmups never disappear from accounting.

### Reconcile missing outcomes or termination evidence

Only after observed prior process/session exit, supply exact identifiers/evidence:

    study.reconcile_attempt( slot=0, attempt=EXACT_ATTEMPT_ID,
        evidence={"termination_confirmed": True, "reason": "Actual observed exit"});
    study.confirm_timeout_termination( slot=0, call="EXACT_CALL_ID",
        evidence={"termination_confirmed": True, "reason": "Actual observed exit"});

Admitted calls without durable output stay spent/unknown. Only one identical
pre-persistence infrastructure retry is eligible after confirmed termination.
Recorded timeouts, wrong, malformed or truncated replies never retry.

If saved bytes predate lost work, report partial recovery. Provide a defensible
spending bound and affected worlds; unbounded unknown spending blocks generation
while allowing reports:

    study.record_lost_spending(
        gap={"resolved": True, "upper": ACTUAL_UPPER_BOUND, "evidence": "Actual evidence"},
        affected_worlds=["main-XX"]);
    # For unbounded loss: resolved=False, upper=None. Never assume zero.

### Complete inspection, annotation and read-only reporting

    study.inspect(world="pilot-00", phase="retain", agent=0, audit_limit=None);
    study.links(all_files=True);
    study.annotate("unique-review-id", "ambiguous", ["exact/call/id"], "Trace-based notes");

Filtering changes only this view. Private submissions, truth and thinking are
audit material, never partner input. Thinking is not a faithful explanation.
No LLM judge is used.

With an initialized controller use study.build_report(). With only intact files,
the same method reconstructs reports read-only without creating workers/CUDA.
Outside the notebook, use its exact saved source:

    python scripts/i01_course.py --source SOURCE --run EXTRACTED status
    python scripts/i01_course.py --source SOURCE --run EXTRACTED report --output REPORT_DIR

A previous checkpoint uses its exact saved source release for inference.
Reports remain readable; a UI update cannot replace its scientific evidence.
